<div align="center">
    <h2><b>Architecture Design Document (ADD)</b></h2>
    <h1>Проектування глобальної E-Commerce платформи (Multi-Region)</h1>
    <h3>Глобальний маркетплейс «2BeMarket»</h3>
    <br>

**Метадані документа:**

| **Тип документа:** | High-Level Design (HLD) & Global Scale |
| :--- | :--- |
| **Статус:** | Draft / В розробці |
| **Масштаб:** | Global (Multi-Region Active-Active) |
| **Версія архітектури:** | 1.0.0 |
| **Дата створення:** | Березень 2026 р. |

<br>

**Автор проекту:**

| **Ім'я:** | Олег Гаценко |
| :--- | :--- |
| **Роль:** | Principal System Architect / Студент магістратури |
| **Організація:** | NeoVersity (Master's in AI & Machine Learning) |

</div>

---

### **Executive Summary (Резюме архітектури):**

Цей документ описує комплексне проектування високорівневої архітектури (HLD) для глобальної платформи електронної комерції **2BeMarket** (аналог Amazon / AliExpress). Платформа розрахована на понад **100 млн DAU** та роботу в умовах екстремальних навантажень (Global Black Friday). Головний архітектурний виклик — забезпечення мілісекундної затримки для покупців по всьому світу зі збереженням суворої консистентності фінансових транзакцій.

**Ключові архітектурні парадигми та рішення:**
* **Multi-Region Active-Active:** Розгортання інфраструктури у трьох незалежних географічних зонах (США, Європа, Азія) за допомогою Global Server Load Balancing (GSLB). Це гарантує виживання системи навіть у разі падіння цілого континенту (Disaster Recovery).
* **Conflict-Free Carts (CRDT):** Кошики користувачів реалізовані на базі безконфліктних структур даних (CRDT) у розподіленому NoSQL-сховищі (DynamoDB/Cassandra), що виключає втрату товарів при перемиканні користувача між регіонами.
* **Distributed Transactions & Saga:** Відмова від повільних 2PC-транзакцій на користь патерну **Saga** (Оркестрація) з використанням `Idempotency Keys`. Це гарантує надійність списання коштів та інвентаризації складських залишків.
* **Global Polyglot Persistence:** Використання NewSQL (наприклад, Google Spanner / CockroachDB) для транзакційного ядра з суворою консистентністю (ACID) на глобальному рівні, та ElasticSearch для миттєвого повнотекстового пошуку товарів.
* **Machine Learning & Personalization:** Асинхронний пайплайн (Kafka + Flink) для збору клікстріму та віддачі персоналізованих рекомендацій у режимі реального часу як для веб-клієнтів, так і для **мобільних застосунків**.
* **Multi-Cloud & Data Sovereignty:** Інфраструктура є Cloud-Agnostic (Kubernetes) і розгорнута в різних хмарах (AWS для Америки, GCP для Європи/Швейцарії, Alibaba для Азії). Це гарантує юридичну відповідність суворим місцевим законам про зберігання персональних даних (GDPR, FADP, PIPL).
* **Edge Security & Geo-Fencing:** Використання Cloudflare WAF на рівні глобального маршрутизатора для захисту від DDoS та жорсткого геоблокування (Geo-Ban) підсанкційних регіонів (наприклад, рф) ще до потрапляння трафіку у внутрішню мережу маркетплейсу.

### **Контекстна діаграма потоків (Level 0):**

```mermaid
graph LR
    %% Styling (Luxury & Dark Theme)
    classDef source fill:#2b2b2b,stroke:#00ffcc,stroke-width:2px,color:#fff
    classDef stream fill:#1a1a1a,stroke:#ff3366,stroke-width:2px,color:#fff
    classDef batch fill:#1a1a1a,stroke:#3399ff,stroke-width:2px,color:#fff
    classDef sink fill:#2b2b2b,stroke:#ffcc00,stroke-width:2px,color:#fff
    classDef banned fill:#3a0000,stroke:#ff0000,stroke-width:2px,color:#fff
    
    %% Нові Люксові Стилі
    classDef buyer_lux fill:#5e35b1,stroke:#ffd700,stroke-width:3px,color:#fff,rx:10px,ry:10px
    classDef seller_lux fill:#00796b,stroke:#ffd700,stroke-width:3px,color:#fff,rx:10px,ry:10px

    Buyer(("📱 Покупці<br>(Мобільний застосунок / Web)")):::buyer_lux
    Seller(("💻 Продавці<br>(Merchant Portal)")):::seller_lux
    Sanctioned(("🚫 Підсанкційний трафік<br>(рф тощо)")):::banned

    WAF{{"🛡️ Edge Security & WAF<br>(Cloudflare / Geo-Ban)"}}:::source
    GLB{{"🌐 Global Load Balancer<br>& API Gateway"}}:::source

    Catalog["🛒 Catalog & Search<br>(ElasticSearch + Redis)"]:::batch
    Checkout["💳 Cart & Checkout<br>(Saga Orchestrator)"]:::stream

    Bus[["⚡ Global Event Bus<br>(Apache Kafka)"]]:::source

    Storage[("🗄️ Polyglot Persistence<br>(NewSQL, Cassandra)")]:::sink
    Bank[["🏦 Платіжні Шлюзи<br>(Stripe/PayPal)"]]:::sink
    Logistics[["📦 Логістичні Хаби<br>(DHL/FedEx/Local Post)"]]:::sink

    %% Зв'язки безпеки та роутингу
    Buyer -- "Мільйони запитів/сек<br>(Пошук, Кошик)" --> WAF
    Seller -- "Управління інвентарем" --> WAF
    Sanctioned -. "Connection Dropped" .-> WAF
    WAF -- "Авторизований трафік" --> GLB

    %% Внутрішні потоки
    GLB -- "Read-heavy трафік" --> Catalog
    GLB -- "Write-heavy (Idempotent)" --> Checkout

    Checkout -- "Синхронна оплата" --> Bank
    Checkout -- "Асинхронні події" --> Bus

    Bus -- "Eventual Consistency" --> Storage
    Checkout -. "ACID Транзакції" .-> Storage
    Catalog -. "Індексація даних" .-> Storage
    Bus -- "Push: Інтеграція доставки" --> Logistics
```